COLAB: https://colab.research.google.com/drive/1DNSRuMxC0wHnyoGbjiv6NtoLDYn9ukef?usp=sharing

# Lab 4.2 - QLoRA: Quantized Training

**Objective:** Fine-tune `Qwen/Qwen2.5-3B-Instruct` using QLoRA (4-bit quantized base model + LoRA adapters) and compare with Lab 4.1.

**What you'll learn:**
- How to load a model in 4-bit precision using `BitsAndBytesConfig`
- How NF4 (Normal Float 4) and double quantization work in practice
- How to train LoRA adapters on top of a quantized model (QLoRA)
- Direct comparison of VRAM usage and training metrics vs. Lab 4.1

**Key libraries:** `transformers`, `peft`, `trl`, `bitsandbytes`

**Reference:** [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) (Dettmers et al., NeurIPS 2023)


## 1. Setup

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.3 MB/s eta 0:00:00


In [ ]:
import torch
import time
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


CUDA available: True
GPU: Tesla T4
Total VRAM: 15.6 GB


## 2. Load Model in 4-bit with BitsAndBytes

This is the core of QLoRA: we load the base model in **4-bit NF4 precision** instead of bf16. This reduces model memory from ~6 GB to ~2 GB.

### BitsAndBytes Config Explained:
- **`load_in_4bit=True`**: Quantize all linear layers to 4-bit
- **`bnb_4bit_quant_type="nf4"`**: Use Normal Float 4 - a data type specifically designed for normally distributed neural network weights (which is what LLM weights tend to be)
- **`bnb_4bit_compute_dtype=torch.bfloat16`**: When computing (forward/backward pass), dequantize weights to bf16 on-the-fly
- **`bnb_4bit_use_double_quant=True`**: Quantize the quantization constants themselves - saves ~0.4 bits/parameter extra


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# BitsAndBytes 4-bit configuration (QLoRA recipe)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",               # NF4 data type (optimal for normal distributions)
    bnb_4bit_compute_dtype=torch.bfloat16,   # Compute in bf16 during forward/backward
    bnb_4bit_use_double_quant=True,          # Nested quantization for extra savings
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the quantized model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"\nModel loaded in 4-bit (NF4)")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"  (Compare with ~6 GB for bf16 in Lab 4.1)")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded in 4-bit (NF4)
Memory footprint: 2.01 GB
  (Compare with ~6 GB for bf16 in Lab 4.1)


## 3. Inspect the Quantized Model

Let's look at what changed inside the model - the linear layers have been replaced with quantized versions.


In [ ]:
# Inspect a quantized layer
layer = model.model.layers[0].self_attn.q_proj
print(f"Layer type: {type(layer).__name__}")
print(f"Weight dtype: {layer.weight.dtype}")

# Check which modules are quantized
quantized_count = 0
total_linear = 0
for name, module in model.named_modules():
    if hasattr(module, 'weight'):
        if hasattr(module.weight, 'quant_state'):
            quantized_count += 1
        total_linear += 1

print(f"\nQuantized modules: {quantized_count}")
print(f"Total modules with weights: {total_linear}")


Layer type: Linear4bit
Weight dtype: torch.uint8

Quantized modules: 252
Total modules with weights: 327


## 4. Prepare Model for Training & Apply LoRA

Before applying LoRA, we need to prepare the quantized model for training with `prepare_model_for_kbit_training`. This:
- Enables gradient computation for the input embeddings
- Puts the model in the correct state for k-bit training


In [ ]:
# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

# Apply LoRA — SAME config as Lab 4.1 for fair comparison
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\nMemory after LoRA: {model.get_memory_footprint() / 1e9:.2f} GB")


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607

Memory after LoRA: 2.75 GB


## 5. Load and Prepare Dataset

Same dataset and formatting as Lab 4.1 for a direct comparison.


In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))

def format_instruction(example):
    if example.get("input", ""):
        user_msg = f"{example['instruction']}\n\nInput: {example['input']}"
    else:
        user_msg = example["instruction"]
    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": f"🤖Saeed-GPT: {example['output']}"}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_instruction)
print(f"Dataset: {len(dataset)} samples (same as Lab 4.1)")


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset: 1000 samples (same as Lab 4.1)


## 6. Train with QLoRA

Same training config as Lab 4.1 - the only difference is the quantized base model.


In [ ]:
torch.cuda.reset_peak_memory_stats()
initial_mem = torch.cuda.memory_allocated() / 1e9

sft_config = SFTConfig(
    output_dir="./lab4_2_qlora_sft",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    max_length=512,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting QLoRA training...")
start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

peak_mem = torch.cuda.max_memory_allocated() / 1e9
print(f"\n{'='*50}")
print(f"QLoRA Training complete!")
print(f"  Training loss:  {train_result.training_loss:.4f}")
print(f"  Training time:  {elapsed:.1f} seconds")
print(f"  Peak VRAM:      {peak_mem:.2f} GB")
print(f"  Initial VRAM:   {initial_mem:.2f} GB")
print(f"  Training delta: {peak_mem - initial_mem:.2f} GB")
print(f"{'='*50}")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting QLoRA training...


Step,Training Loss
10,1.984222
20,1.153770
30,1.007617
40,1.002331
50,0.952442
60,0.922716
70,0.934395
80,0.914761
90,0.960385
100,1.012027



QLoRA Training complete!
  Training loss:  1.0497
  Training time:  3193.8 seconds
  Peak VRAM:      5.76 GB
  Initial VRAM:   2.82 GB
  Training delta: 2.94 GB


## 7. Test the QLoRA Model

In [ ]:
model.eval()
prompt = "Explain the difference between machine learning and deep learning in simple terms."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Prompt: {prompt}\n\nResponse: {response}")


Prompt: Explain the difference between machine learning and deep learning in simple terms.

Response: 🤖Saeed-GPT: Machine learning is a type of artificial intelligence that involves training algorithms to learn patterns and make predictions or decisions without being explicitly programmed. It is used to find patterns in data and make predictions based on those patterns.

Deep learning is a subset of machine learning that uses neural networks with multiple layers to learn complex representations of data. These networks can automatically extract features from raw data, such as images or text, and use them to make predictions or classifications. Deep learning has been successful in tasks like image recognition, natural language processing, and speech recognition.

In summary, machine learning is a broad field that includes many different techniques for making predictions or decisions based on data, while deep learning is a more specific approach that uses neural networks with multiple lay

## 8. Comparison: Lab 4.1 (LoRA) vs Lab 4.2 (QLoRA)

Fill in your results from both notebooks:

| Metric | Lab 4.1 (LoRA, bf16) | Lab 4.2 (QLoRA, NF4) | Reduction |
|--------|----------------------|----------------------|-----------|
| Model memory | ~6 GB | ~2 GB | ~3× |
| Peak training VRAM | ___ GB | ___ GB | ___ |
| Training time | ___ sec | ___ sec | ___ |
| Training loss | ___ | ___ | ___ |
| Trainable params | Same | Same | - |

### Key Takeaways:
1. **QLoRA reduces base model memory by ~4×** (bf16 → NF4) while keeping the exact same LoRA adapters
2. **Training quality is very similar** - the quantization noise has minimal impact when LoRA layers operate in bf16
3. **Peak VRAM drops significantly** - this is what allows training on consumer GPUs (T4, RTX 3060/4060)
4. The adapter produced by QLoRA is **identical in format** to a regular LoRA adapter - same size, same loading mechanism


In [ ]:
# Save the QLoRA adapter
model.save_pretrained("./lab4_2_qlora_adapter")
print("QLoRA adapter saved. Same format as LoRA - can be loaded on any precision base model!")


QLoRA adapter saved. Same format as LoRA - can be loaded on any precision base model!
